# day_aggs_v1 fundamental signal demo

这个 Notebook 用来复现和检查 `examples/day_aggs_v1_fundamental_signal.yaml` 的运行结果。

默认行为是直接读取已经落盘的结果文件。把下一格里的 `force_rerun` 改成 `True`，可以重新全量执行一次组合信号流水线。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

module_root = Path('/home/yluel/share/projects/quantsociety_backend_project/strategy_layer/multiple_factor_composite')
if str(module_root) not in sys.path:
    sys.path.insert(0, str(module_root))

from pipeline import run_from_config

config_path = module_root / 'examples' / 'day_aggs_v1_fundamental_signal.yaml'
output_root = module_root / 'runs' / 'day_aggs_v1_fundamental_signal_v1'
signal_path = output_root / 'signals' / 'composite_signal.parquet'
manifest_path = output_root / 'manifest.json'
weight_path = output_root / 'weights' / 'weight_history.parquet'

print(f'config_path={config_path}')
print(f'output_root={output_root}')

config_path=/home/yluel/share/projects/quantsociety_backend_project/strategy_layer/multiple_factor_composite/examples/day_aggs_v1_fundamental_signal.yaml
output_root=/home/yluel/share/projects/quantsociety_backend_project/strategy_layer/multiple_factor_composite/runs/day_aggs_v1_fundamental_signal_v1


In [2]:
force_rerun = False

if force_rerun or not signal_path.exists():
    result = run_from_config(config_path)
    print(json.dumps({
        'signal_rows': len(result['signal']),
        'outputs': result['outputs'],
    }, ensure_ascii=False, indent=2))
else:
    manifest = json.loads(manifest_path.read_text())
    print('Using existing outputs:')
    print(json.dumps(manifest, ensure_ascii=False, indent=2))

Using existing outputs:
{
  "signal_id": "day_aggs_v1_fundamental_signal",
  "version": "v1",
  "factor_ids": [
    "day_aggs_v1_fundamental_asset_scale_rank_2016_2025_v1",
    "day_aggs_v1_fundamental_book_strength_rank_2016_2025_v1",
    "day_aggs_v1_fundamental_cash_reinvestment_rank_2016_2025_v1",
    "day_aggs_v1_fundamental_float_tightness_rank_2016_2025_v1"
  ],
  "factor_names": [
    "asset_scale",
    "book_strength",
    "cash_reinvestment",
    "float_tightness"
  ],
  "composed_factor_names": [
    "asset_scale",
    "book_strength",
    "cash_reinvestment",
    "float_tightness"
  ],
  "row_count": 10608981,
  "output_files": {
    "raw_panel": "/home/yluel/share/projects/quantsociety_backend_project/strategy_layer/multiple_factor_composite/runs/day_aggs_v1_fundamental_signal_v1/panels/raw_factor_panel.parquet",
    "preprocessed_panel": "/home/yluel/share/projects/quantsociety_backend_project/strategy_layer/multiple_factor_composite/runs/day_aggs_v1_fundamental_signal_v1

In [3]:
signal = pd.read_parquet(signal_path)
summary = {
    'rows': int(len(signal)),
    'unique_dates': int(signal['datetime'].nunique()),
    'unique_assets': int(signal['asset'].nunique()),
    'start': str(signal['datetime'].min()),
    'end': str(signal['datetime'].max()),
    'selected_rows': int(signal['selected_flag'].sum()),
}
summary

{'rows': 10608981,
 'unique_dates': 2513,
 'unique_assets': 10140,
 'start': '2016-01-05 05:00:00',
 'end': '2025-12-31 05:00:00',
 'selected_rows': 125494}

In [4]:
signal.head(10)

,datetime,asset,composite_score,rank,selected_flag,side,signal_id,signal_version
0,2016-01-05 05:00:00,MCS,0.000000,1.0,True,LONG,day_aggs_v1_fundamental_signal,v1
1,2016-01-06 05:00:00,CAG,1.767736,1.0,True,LONG,day_aggs_v1_fundamental_signal,v1
2,2016-01-06 05:00:00,MLHR,1.149618,2.0,True,LONG,day_aggs_v1_fundamental_signal,v1
3,2016-01-06 05:00:00,DRI,0.096033,3.0,True,LONG,day_aggs_v1_fundamental_signal,v1
4,2016-01-06 05:00:00,RAD,-0.111336,4.0,True,LONG,day_aggs_v1_fundamental_signal,v1
5,2016-01-06 05:00:00,MCS,-0.279624,5.0,True,LONG,day_aggs_v1_fundamental_signal,v1
6,2016-01-06 05:00:00,RPM,-0.312239,6.0,True,LONG,day_aggs_v1_fundamental_signal,v1
7,2016-01-06 05:00:00,PCYO,-0.568705,7.0,True,LONG,day_aggs_v1_fundamental_signal,v1
8,2016-01-06 05:00:00,PIR,-1.741482,8.0,True,LONG,day_aggs_v1_fundamental_signal,v1
9,2016-01-07 05:00:00,UNF,1.722264,1.0,True,LONG,day_aggs_v1_fundamental_signal,v1


In [5]:
daily_selected = signal.groupby('datetime')['selected_flag'].sum()
weights = pd.read_parquet(weight_path)

print(daily_selected.describe())
weights.head()

count    2513.000000
mean       49.937923
std         1.477581
min         1.000000
25%        50.000000
50%        50.000000
75%        50.000000
max        50.000000
Name: selected_flag, dtype: float64


,datetime,asset_scale,book_strength,cash_reinvestment,float_tightness
0,2016-01-05 05:00:00,0.25,0.25,0.25,0.25
1,2016-01-06 05:00:00,0.25,0.25,0.25,0.25
2,2016-01-07 05:00:00,0.25,0.25,0.25,0.25
3,2016-01-08 05:00:00,0.25,0.25,0.25,0.25
4,2016-01-11 05:00:00,0.25,0.25,0.25,0.25
